In [3]:

import torch
import numpy as np
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

print("\nEnvironment check completed!")


PyTorch version: 2.15.0+cpu
CUDA available: False
Running on CPU

Environment check completed!


In [4]:

from pathlib import Path
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import torch
import numpy as np

# Processed dataset location
DATA_ROOT = Path(
    r"D:\brain tumour\brisc2025\segmentation_processed_256"
)

TRAIN_IMAGE_DIR = DATA_ROOT / "train" / "images"
TRAIN_MASK_DIR = DATA_ROOT / "train" / "masks"

TEST_IMAGE_DIR = DATA_ROOT / "test" / "images"
TEST_MASK_DIR = DATA_ROOT / "test" / "masks"

BATCH_SIZE = 8
RANDOM_SEED = 42

print("Dataset paths configured!")
print("Training images:", len(list(TRAIN_IMAGE_DIR.glob("*.png"))))
print("Testing images:", len(list(TEST_IMAGE_DIR.glob("*.png"))))


Dataset paths configured!
Training images: 3933
Testing images: 860


In [5]:

class BrainTumorDataset(Dataset):

    def __init__(self, image_dir, mask_dir):

        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)

        self.image_files = sorted(
            self.image_dir.glob("*.png")
        )

        # Verify matching masks
        for image_path in self.image_files:

            mask_path = self.mask_dir / image_path.name

            if not mask_path.exists():
                raise FileNotFoundError(
                    f"Mask missing: {mask_path}"
                )

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, index):

        image_path = self.image_files[index]
        mask_path = self.mask_dir / image_path.name

        # Load MRI and mask
        image = np.array(
            Image.open(image_path).convert("L"),
            dtype=np.float32
        ) / 255.0

        mask = np.array(
            Image.open(mask_path).convert("L"),
            dtype=np.float32
        ) / 255.0

        # Convert to PyTorch tensors
        image = torch.from_numpy(image).unsqueeze(0)
        mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask


print("BrainTumorDataset class created!")


BrainTumorDataset class created!


In [6]:

train_dataset = BrainTumorDataset(
    TRAIN_IMAGE_DIR,
    TRAIN_MASK_DIR
)

test_dataset = BrainTumorDataset(
    TEST_IMAGE_DIR,
    TEST_MASK_DIR
)

print("Training samples:", len(train_dataset))
print("Testing samples:", len(test_dataset))

# Load one sample
sample_image, sample_mask = train_dataset[0]

print("\nSample image shape:", sample_image.shape)
print("Sample mask shape:", sample_mask.shape)

print("Image dtype:", sample_image.dtype)
print("Mask dtype:", sample_mask.dtype)

print("Image range:", sample_image.min().item(),
      sample_image.max().item())

print("Mask unique values:", torch.unique(sample_mask))


Training samples: 3933
Testing samples: 860

Sample image shape: torch.Size([1, 256, 256])
Sample mask shape: torch.Size([1, 256, 256])
Image dtype: torch.float32
Mask dtype: torch.float32
Image range: 0.0 0.9215686321258545
Mask unique values: tensor([0., 1.])


In [7]:
# Calculate training and validation sizes
train_size = int(0.8 * len(train_dataset))
val_size = len(train_dataset) - train_size

# Reproducible random split
generator = torch.Generator().manual_seed(RANDOM_SEED)

train_subset, val_subset = random_split(
    train_dataset,
    [train_size, val_size],
    generator=generator
)

print("Training subset:", len(train_subset))
print("Validation subset:", len(val_subset))
print("Testing dataset:", len(test_dataset))

Training subset: 3146
Validation subset: 787
Testing dataset: 860


In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("DataLoaders created successfully!")

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

DataLoaders created successfully!
Training batches: 394
Validation batches: 99
Testing batches: 108


In [9]:
images, masks = next(iter(train_loader))

print("Images batch shape:", images.shape)
print("Masks batch shape:", masks.shape)

Images batch shape: torch.Size([8, 1, 256, 256])
Masks batch shape: torch.Size([8, 1, 256, 256])


In [10]:
import torch
import torch.nn as nn

In [11]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [12]:
test_block = DoubleConv(1, 64)

sample = torch.randn(1, 1, 256, 256)

output = test_block(sample)

print("Input shape:", sample.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([1, 1, 256, 256])
Output shape: torch.Size([1, 64, 256, 256])


In [13]:
class Encoder(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2)

    def forward(self, x):

        features = self.conv(x)
        pooled = self.pool(features)

        return features, pooled

In [14]:
encoder1 = Encoder(1, 64)

sample = torch.randn(1, 1, 256, 256)

features, pooled = encoder1(sample)

print("Input shape:", sample.shape)
print("Features shape:", features.shape)
print("Pooled shape:", pooled.shape)

Input shape: torch.Size([1, 1, 256, 256])
Features shape: torch.Size([1, 64, 256, 256])
Pooled shape: torch.Size([1, 64, 128, 128])


In [15]:
class UNetEncoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.enc1 = Encoder(1, 64)
        self.enc2 = Encoder(64, 128)
        self.enc3 = Encoder(128, 256)
        self.enc4 = Encoder(256, 512)

        self.bottleneck = DoubleConv(512, 1024)

    def forward(self, x):

        f1, p1 = self.enc1(x)
        f2, p2 = self.enc2(p1)
        f3, p3 = self.enc3(p2)
        f4, p4 = self.enc4(p3)

        bottleneck = self.bottleneck(p4)

        return f1, f2, f3, f4, bottleneck

In [16]:
encoder = UNetEncoder()

sample = torch.randn(1, 1, 256, 256)

f1, f2, f3, f4, bottleneck = encoder(sample)

print("Encoder Level 1:", f1.shape)
print("Encoder Level 2:", f2.shape)
print("Encoder Level 3:", f3.shape)
print("Encoder Level 4:", f4.shape)
print("Bottleneck:", bottleneck.shape)

Encoder Level 1: torch.Size([1, 64, 256, 256])
Encoder Level 2: torch.Size([1, 128, 128, 128])
Encoder Level 3: torch.Size([1, 256, 64, 64])
Encoder Level 4: torch.Size([1, 512, 32, 32])
Bottleneck: torch.Size([1, 1024, 16, 16])


In [17]:
class Decoder(nn.Module):

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(
            out_channels + skip_channels,
            out_channels
        )

    def forward(self, x, skip):

        x = self.up(x)

        x = torch.cat([x, skip], dim=1)

        x = self.conv(x)

        return x

In [18]:
decoder1 = Decoder(
    in_channels=1024,
    skip_channels=512,
    out_channels=512
)

bottleneck_sample = torch.randn(1, 1024, 16, 16)
skip_sample = torch.randn(1, 512, 32, 32)

output = decoder1(bottleneck_sample, skip_sample)

print("Bottleneck shape:", bottleneck_sample.shape)
print("Skip connection shape:", skip_sample.shape)
print("Decoder output shape:", output.shape)

Bottleneck shape: torch.Size([1, 1024, 16, 16])
Skip connection shape: torch.Size([1, 512, 32, 32])
Decoder output shape: torch.Size([1, 512, 32, 32])


In [19]:
class UNetDecoder(nn.Module):

    def __init__(self):
        super().__init__()

        self.dec1 = Decoder(1024, 512, 512)
        self.dec2 = Decoder(512, 256, 256)
        self.dec3 = Decoder(256, 128, 128)
        self.dec4 = Decoder(128, 64, 64)

    def forward(self, bottleneck, f1, f2, f3, f4):

        d1 = self.dec1(bottleneck, f4)
        d2 = self.dec2(d1, f3)
        d3 = self.dec3(d2, f2)
        d4 = self.dec4(d3, f1)

        return d4

In [20]:
decoder = UNetDecoder()

decoder_output = decoder(
    bottleneck,
    f1,
    f2,
    f3,
    f4
)

print("Final decoder output:", decoder_output.shape)

Final decoder output: torch.Size([1, 64, 256, 256])


In [21]:
class OutputLayer(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):
        return self.conv(x)

In [22]:
output_layer = OutputLayer(64, 1)

segmentation_output = output_layer(decoder_output)

print("Decoder output:", decoder_output.shape)
print("Segmentation output:", segmentation_output.shape)

Decoder output: torch.Size([1, 64, 256, 256])
Segmentation output: torch.Size([1, 1, 256, 256])


In [23]:
class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = UNetEncoder()

        # Decoder
        self.decoder = UNetDecoder()

        # Final output layer
        self.output_layer = OutputLayer(64, 1)

    def forward(self, x):

        # Extract encoder features
        f1, f2, f3, f4, bottleneck = self.encoder(x)

        # Reconstruct spatial dimensions
        decoder_output = self.decoder(
            bottleneck, f1, f2, f3, f4
        )

        # Generate segmentation logits
        output = self.output_layer(decoder_output)

        return output

In [24]:
model = UNet()

sample_input = torch.randn(1, 1, 256, 256)

with torch.no_grad():
    prediction = model(sample_input)

print("Input shape:", sample_input.shape)
print("Prediction shape:", prediction.shape)

total_params = sum(
    parameter.numel() for parameter in model.parameters()
)

print("Total parameters:", f"{total_params:,}")

Input shape: torch.Size([1, 1, 256, 256])
Prediction shape: torch.Size([1, 1, 256, 256])
Total parameters: 31,030,593


In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DiceBCELoss(nn.Module):

    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, predictions, targets):

        # BCE Loss
        bce_loss = self.bce(predictions, targets)

        # Dice Loss
        predictions = torch.sigmoid(predictions)

        predictions = predictions.view(-1)
        targets = targets.view(-1)

        smooth = 1e-6

        intersection = (predictions * targets).sum()

        dice_score = (
            (2.0 * intersection + smooth) /
            (predictions.sum() + targets.sum() + smooth)
        )

        dice_loss = 1 - dice_score

        # Combined Loss
        total_loss = bce_loss + dice_loss

        return total_loss

In [26]:
criterion = DiceBCELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function initialized!")
print("Optimizer initialized!")

Loss function initialized!
Optimizer initialized!


In [27]:
images, masks = next(iter(train_loader))

with torch.no_grad():
    predictions = model(images)

    loss = criterion(predictions, masks)

print("Predictions shape:", predictions.shape)
print("Masks shape:", masks.shape)
print("Test loss:", loss.item())

Predictions shape: torch.Size([8, 1, 256, 256])
Masks shape: torch.Size([8, 1, 256, 256])
Test loss: 1.6802542209625244


In [28]:
import time

NUM_EPOCHS = 20
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(DEVICE)

print("Training device:", DEVICE)
print("Number of epochs:", NUM_EPOCHS)
print("Model moved to device!")

Training device: cpu
Number of epochs: 20
Model moved to device!


In [29]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()
    running_loss = 0.0

    for images, masks in loader:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        predictions = model(images)
        loss = criterion(predictions, masks)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(loader)

    return epoch_loss

In [30]:
def validate_one_epoch(model, loader, criterion, device):

    model.eval()
    running_loss = 0.0

    with torch.no_grad():

        for images, masks in loader:

            images = images.to(device)
            masks = masks.to(device)

            predictions = model(images)
            loss = criterion(predictions, masks)

            running_loss += loss.item()

    epoch_loss = running_loss / len(loader)

    return epoch_loss